## 7. 总结

本教程介绍了音频特征提取的核心概念：

| 特征 | 维度 | 优点 | 应用场景 |
|:-----|:-----|:-----|:---------|
| 波形 | 1D | 无信息损失 | 端到端模型 |
| STFT | 2D | 时频信息完整 | 音频分析 |
| Mel 频谱 | 2D | 符合人耳感知 | ASR/TTS |
| MFCC | 1D | 紧凑、去相关 | 传统 ASR |

**下一步**: 
- 学习 Whisper 语音识别模型
- 学习 TTS 文本转语音模型

In [ ]:
# 使用统一接口提取不同特征
extractor = AudioFeatureExtractor(config)

# 提取不同类型的特征
mel_feat = extractor(waveform_tensor, feature_type="mel")
log_mel_feat = extractor(waveform_tensor, feature_type="log_mel")
mfcc_feat = extractor(waveform_tensor, feature_type="mfcc")

print(f"Mel 特征形状: {mel_feat.shape}")
print(f"Log-Mel 特征形状: {log_mel_feat.shape}")
print(f"MFCC 特征形状: {mfcc_feat.shape}")

In [ ]:
from audio_features import AudioFeatureExtractor, create_audio_extractor

# 使用预设配置创建提取器
whisper_extractor = create_audio_extractor("whisper")
asr_extractor = create_audio_extractor("asr")
tts_extractor = create_audio_extractor("tts")

print("Whisper 配置:")
print(f"  采样率: {whisper_extractor.config.sample_rate}")
print(f"  Mel bins: {whisper_extractor.config.n_mels}")

print("\nASR 配置:")
print(f"  采样率: {asr_extractor.config.sample_rate}")
print(f"  Mel bins: {asr_extractor.config.n_mels}")

print("\nTTS 配置:")
print(f"  采样率: {tts_extractor.config.sample_rate}")
print(f"  Mel bins: {tts_extractor.config.n_mels}")

## 6. 统一特征提取器

`AudioFeatureExtractor` 提供了一个统一的接口来提取各种音频特征：

In [ ]:
from audio_features import SpecAugment

# 创建 SpecAugment 模块
config_aug = AudioConfig(
    freq_mask_param=15,   # 最大频率遮蔽宽度
    time_mask_param=35,   # 最大时间遮蔽宽度
    n_freq_masks=2,       # 频率遮蔽次数
    n_time_masks=2        # 时间遮蔽次数
)
spec_augment = SpecAugment(config_aug)

# 应用 SpecAugment
original_spec = log_mel_output.clone()
augmented_spec = spec_augment(original_spec.clone(), training=True)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].imshow(original_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_xlabel('Time Frame')
axes[0].set_ylabel('Mel Bin')
axes[0].set_title('Original Log-Mel Spectrogram')

axes[1].imshow(augmented_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_xlabel('Time Frame')
axes[1].set_ylabel('Mel Bin')
axes[1].set_title('After SpecAugment (Time & Frequency Masking)')

plt.tight_layout()
plt.show()

print("黑色区域是被遮蔽的部分")
print("频率遮蔽: 水平条带")
print("时间遮蔽: 垂直条带")

## 5. SpecAugment 数据增强

SpecAugment 是一种简单有效的频谱数据增强方法，通过在频谱图上应用遮蔽来提高模型的鲁棒性：

1. **时间遮蔽 (Time Masking)**: 随机遮蔽连续的时间帧
2. **频率遮蔽 (Frequency Masking)**: 随机遮蔽连续的频率 bin

这种方法在语音识别任务中非常有效，可以显著提高模型的泛化能力。

In [ ]:
# 计算 MFCC
config_mfcc = AudioConfig(sample_rate=sample_rate, n_fft=512, hop_length=128, n_mels=80, n_mfcc=13)
mfcc_extractor = MFCC(config_mfcc)

mfcc_output = mfcc_extractor(waveform_tensor)
mfcc_with_deltas = mfcc_extractor(waveform_tensor, include_deltas=True)

print(f"MFCC 形状: {mfcc_output.shape}")
print(f"MFCC + Delta + Delta-Delta 形状: {mfcc_with_deltas.shape}")

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# MFCC
im1 = axes[0].imshow(mfcc_output[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_xlabel('Time Frame')
axes[0].set_ylabel('MFCC Coefficient')
axes[0].set_title('MFCC (13 coefficients)')
plt.colorbar(im1, ax=axes[0])

# MFCC + Deltas
im2 = axes[1].imshow(mfcc_with_deltas[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_xlabel('Time Frame')
axes[1].set_ylabel('Coefficient')
axes[1].set_title('MFCC + Delta + Delta-Delta (39 coefficients)')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("\n前 13 个: 静态 MFCC")
print("中间 13 个: 一阶差分 (Delta) - 捕捉变化速度")
print("后 13 个: 二阶差分 (Delta-Delta) - 捕捉变化加速度")

In [ ]:
from audio_features import MFCC, create_dct_matrix

# 可视化 DCT 矩阵
dct_matrix = create_dct_matrix(n_mfcc=13, n_mels=80)
print(f"DCT 矩阵形状: {dct_matrix.shape}")

plt.figure(figsize=(12, 4))
plt.imshow(dct_matrix.numpy(), aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel('Mel Bin')
plt.ylabel('MFCC Coefficient')
plt.title('DCT Matrix (13 MFCCs from 80 Mel bins)')
plt.tight_layout()
plt.show()

# 音频特征提取教程

本教程介绍音频信号处理和特征提取的核心概念，包括：

1. **音频信号基础** - 采样、量化、时域与频域
2. **短时傅里叶变换 (STFT)** - 时频分析的基础
3. **Mel 频谱图** - 符合人耳感知的频谱表示
4. **MFCC** - 梅尔频率倒谱系数
5. **SpecAugment** - 频谱数据增强

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

# 添加 src 目录到路径
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import numpy as np
import matplotlib.pyplot as plt

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 10

# 检查设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 音频信号基础

### 1.1 数字音频表示

音频信号是连续的模拟信号，需要通过**采样**和**量化**转换为数字信号：

```
模拟信号 → 采样 (Sampling) → 量化 (Quantization) → 数字信号
```

**关键参数**:
- **采样率 (Sample Rate)**: 每秒采样次数，常见值：16kHz (语音)、44.1kHz (音乐)
- **位深度 (Bit Depth)**: 每个采样点的量化精度，常见值：16-bit、24-bit

In [ ]:
# 生成一个简单的正弦波音频信号
sample_rate = 16000  # 16kHz 采样率
duration = 0.1  # 0.1 秒
frequency = 440  # 440Hz (A4 音符)

t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
waveform = np.sin(2 * np.pi * frequency * t)

# 可视化
plt.figure(figsize=(12, 3))
plt.plot(t * 1000, waveform)  # 转换为毫秒
plt.xlabel('Time (ms)')
plt.ylabel('Amplitude')
plt.title(f'Sine Wave: {frequency}Hz at {sample_rate}Hz Sample Rate')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"采样率: {sample_rate} Hz")
print(f"采样点数: {len(waveform)}")
print(f"时长: {duration * 1000} ms")

### 1.2 复合信号

真实的音频信号通常是多个频率成分的叠加：

In [ ]:
# 生成复合信号 (多个频率叠加)
duration = 0.05
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# 基频 + 谐波
f1, f2, f3 = 220, 440, 880  # 基频和两个谐波
signal = 0.5 * np.sin(2 * np.pi * f1 * t) + \
         0.3 * np.sin(2 * np.pi * f2 * t) + \
         0.2 * np.sin(2 * np.pi * f3 * t)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 时域
axes[0].plot(t * 1000, signal)
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Time Domain: Composite Signal')

# 频域 (FFT)
fft_result = np.fft.fft(signal)
freqs = np.fft.fftfreq(len(signal), 1/sample_rate)
positive_mask = freqs >= 0

axes[1].plot(freqs[positive_mask], np.abs(fft_result[positive_mask]))
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')
axes[1].set_title('Frequency Domain: FFT')
axes[1].set_xlim(0, 1500)

plt.tight_layout()
plt.show()

print(f"频率成分: {f1}Hz, {f2}Hz, {f3}Hz")

## 2. 短时傅里叶变换 (STFT)

由于音频信号是**非平稳**的（频率成分随时间变化），我们使用 STFT 分析局部频率特性：

$$X(m, k) = \sum_{n=0}^{N-1} x[n + mH] \cdot w[n] \cdot e^{-j2\pi kn/N}$$

其中：
- $m$: 帧索引
- $k$: 频率 bin 索引
- $H$: 帧移 (hop length)
- $w[n]$: 窗函数

In [ ]:
from audio_features import STFT, AudioConfig

# 生成一个频率随时间变化的信号 (chirp)
duration = 1.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# 线性调频信号: 频率从 200Hz 变化到 2000Hz
f_start, f_end = 200, 2000
phase = 2 * np.pi * (f_start * t + (f_end - f_start) * t**2 / (2 * duration))
chirp_signal = np.sin(phase)

# 转换为 PyTorch tensor
waveform_tensor = torch.tensor(chirp_signal, dtype=torch.float32).unsqueeze(0)

# 创建 STFT 模块
stft = STFT(n_fft=512, hop_length=128, win_length=512)
stft_output = stft(waveform_tensor)

print(f"输入波形形状: {waveform_tensor.shape}")
print(f"STFT 输出形状: {stft_output.shape}")
print(f"  - Batch: {stft_output.shape[0]}")
print(f"  - 频率 bins: {stft_output.shape[1]} (n_fft/2 + 1)")
print(f"  - 时间帧: {stft_output.shape[2]}")

In [ ]:
# 可视化 STFT 频谱图
magnitude = stft_output[0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 时域波形
axes[0].plot(t, chirp_signal)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Chirp Signal (200Hz → 2000Hz)')

# STFT 频谱图
time_axis = np.arange(magnitude.shape[1]) * 128 / sample_rate
freq_axis = np.arange(magnitude.shape[0]) * sample_rate / 512

im = axes[1].imshow(
    20 * np.log10(magnitude + 1e-9),
    aspect='auto',
    origin='lower',
    extent=[time_axis[0], time_axis[-1], freq_axis[0], freq_axis[-1]],
    cmap='viridis'
)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_title('STFT Spectrogram (dB)')
plt.colorbar(im, ax=axes[1], label='dB')

plt.tight_layout()
plt.show()

### 2.1 STFT 参数的影响

- **n_fft**: FFT 点数，决定频率分辨率
- **hop_length**: 帧移，决定时间分辨率
- **win_length**: 窗长，通常等于 n_fft

In [ ]:
# 比较不同 n_fft 的效果
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

n_fft_values = [256, 512, 1024]

for ax, n_fft in zip(axes, n_fft_values):
    stft_module = STFT(n_fft=n_fft, hop_length=n_fft//4)
    output = stft_module(waveform_tensor)
    magnitude = output[0].numpy()
    
    time_axis = np.arange(magnitude.shape[1]) * (n_fft//4) / sample_rate
    freq_axis = np.arange(magnitude.shape[0]) * sample_rate / n_fft
    
    im = ax.imshow(
        20 * np.log10(magnitude + 1e-9),
        aspect='auto',
        origin='lower',
        extent=[time_axis[0], time_axis[-1], freq_axis[0], freq_axis[-1]],
        cmap='viridis'
    )
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'n_fft={n_fft}')
    ax.set_ylim(0, 3000)

plt.suptitle('Effect of n_fft on Time-Frequency Resolution', y=1.02)
plt.tight_layout()
plt.show()

print("n_fft 越大 → 频率分辨率越高，时间分辨率越低")
print("n_fft 越小 → 时间分辨率越高，频率分辨率越低")

## 3. Mel 频谱图

### 3.1 Mel 尺度

人耳对频率的感知是**非线性**的：低频区分辨率高，高频区分辨率低。

**Mel 尺度**模拟这种特性：

$$m = 2595 \cdot \log_{10}(1 + \frac{f}{700})$$

In [ ]:
from audio_features import hz_to_mel, mel_to_hz

# Hz 到 Mel 的转换
frequencies = torch.linspace(0, 8000, 100)
mel_values = hz_to_mel(frequencies)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Hz vs Mel
axes[0].plot(frequencies.numpy(), mel_values.numpy())
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Mel')
axes[0].set_title('Hz to Mel Conversion')
axes[0].grid(True)

# 标注一些关键频率
key_freqs = [100, 500, 1000, 2000, 4000, 8000]
for f in key_freqs:
    m = hz_to_mel(torch.tensor(float(f))).item()
    axes[0].axvline(f, color='r', linestyle='--', alpha=0.3)
    axes[0].annotate(f'{f}Hz\n{m:.0f}mel', (f, m), fontsize=8)

# Mel 尺度上的均匀分布
n_mels = 10
mel_points = torch.linspace(0, hz_to_mel(torch.tensor(8000.0)), n_mels + 2)
hz_points = mel_to_hz(mel_points)

axes[1].bar(range(len(hz_points)), hz_points.numpy(), color='steelblue')
axes[1].set_xlabel('Mel Filter Index')
axes[1].set_ylabel('Center Frequency (Hz)')
axes[1].set_title('Mel Filter Center Frequencies')

plt.tight_layout()
plt.show()

print("Mel 滤波器在低频区更密集，高频区更稀疏")

### 3.2 Mel 滤波器组

In [ ]:
from audio_features import create_mel_filterbank

# 创建 Mel 滤波器组
n_fft = 512
n_mels = 40
filterbank = create_mel_filterbank(n_fft, n_mels, sample_rate)

print(f"滤波器组形状: {filterbank.shape}")
print(f"  - Mel 滤波器数量: {filterbank.shape[0]}")
print(f"  - 频率 bins: {filterbank.shape[1]}")

# 可视化滤波器组
freq_axis = np.arange(filterbank.shape[1]) * sample_rate / n_fft

plt.figure(figsize=(12, 5))
for i in range(0, n_mels, 4):  # 每隔 4 个显示一个
    plt.plot(freq_axis, filterbank[i].numpy(), label=f'Filter {i}')

plt.xlabel('Frequency (Hz)')
plt.ylabel('Weight')
plt.title('Mel Filterbank (showing every 4th filter)')
plt.legend(loc='upper right', fontsize=8)
plt.xlim(0, 8000)
plt.tight_layout()
plt.show()

### 3.3 计算 Mel 频谱图

In [ ]:
from audio_features import MelSpectrogram, LogMelSpectrogram

# 生成模拟语音信号 (多个频率成分随时间变化)
duration = 2.0
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# 模拟语音: 基频 + 谐波 + 噪声
f0 = 150  # 基频
speech = 0.5 * np.sin(2 * np.pi * f0 * t)
speech += 0.3 * np.sin(2 * np.pi * 2 * f0 * t)  # 第一谐波
speech += 0.2 * np.sin(2 * np.pi * 3 * f0 * t)  # 第二谐波
speech += 0.1 * np.random.randn(len(t))  # 噪声

# 添加振幅包络 (模拟语音的起伏)
envelope = 0.5 + 0.5 * np.sin(2 * np.pi * 2 * t)
speech = speech * envelope

waveform_tensor = torch.tensor(speech, dtype=torch.float32).unsqueeze(0)

# 创建 Mel 频谱图提取器
config = AudioConfig(sample_rate=sample_rate, n_fft=512, hop_length=128, n_mels=80)
mel_spec = MelSpectrogram(config)
log_mel_spec = LogMelSpectrogram(config)

# 计算
mel_output = mel_spec(waveform_tensor)
log_mel_output = log_mel_spec(waveform_tensor)

print(f"Mel 频谱图形状: {mel_output.shape}")
print(f"Log-Mel 频谱图形状: {log_mel_output.shape}")

In [ ]:
# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 波形
axes[0, 0].plot(t, speech)
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].set_title('Waveform')

# STFT 频谱图
stft_module = STFT(n_fft=512, hop_length=128)
stft_out = stft_module(waveform_tensor)
axes[0, 1].imshow(
    20 * np.log10(stft_out[0].numpy() + 1e-9),
    aspect='auto', origin='lower', cmap='viridis'
)
axes[0, 1].set_xlabel('Time Frame')
axes[0, 1].set_ylabel('Frequency Bin')
axes[0, 1].set_title('STFT Spectrogram')

# Mel 频谱图
axes[1, 0].imshow(
    mel_output[0].numpy(),
    aspect='auto', origin='lower', cmap='viridis'
)
axes[1, 0].set_xlabel('Time Frame')
axes[1, 0].set_ylabel('Mel Bin')
axes[1, 0].set_title('Mel Spectrogram')

# Log-Mel 频谱图
im = axes[1, 1].imshow(
    log_mel_output[0].numpy(),
    aspect='auto', origin='lower', cmap='viridis'
)
axes[1, 1].set_xlabel('Time Frame')
axes[1, 1].set_ylabel('Mel Bin')
axes[1, 1].set_title('Log-Mel Spectrogram')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

## 4. MFCC (梅尔频率倒谱系数)

MFCC 是语音识别中最常用的特征，通过对 Log-Mel 频谱做 DCT (离散余弦变换) 得到：

```
Mel 频谱 → 对数 → DCT → MFCC
```

**DCT 的作用**:
- 去相关：将相关的 Mel 频率 bin 转换为不相关的系数
- 压缩：大部分能量集中在前几个系数
- 通常只取前 13 个系数